# Financial QA — LoRA vs QLoRA

Fine-tunes Qwen2.5-1.5B-Instruct on `virattt/financial-qa-10K` (10-K filing passages with
questions and reference answers) and measures **ROUGE-1** against the zero-shot base on a
held-out, context-disjoint test split, plus **training time** for the arm.

Run this notebook **once per arm** on the **same GPU type** (T4) -- set `ARM` below -- then
download both zips and run `scripts/compare_arms.py` locally. Training time is only
comparable if both arms ran on the same hardware.

Sidebar: Accelerator -> GPU T4 (x2 is fine; one GPU is used), Internet -> On.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # QLoRA + DataParallel corrupts 4-bit weights

REPO_URL = "https://github.com/shashi05shankar/Text-to-SQL-LoRA-Supervised-Fine-Tuning.git"
BRANCH = "financial-qa-peft"  # must be pushed to the remote before this notebook can clone it

import torch
print("torch", torch.__version__, "| visible GPUs:", torch.cuda.device_count(), "(must be 1)")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4.")
print(torch.cuda.get_device_name(0))


In [ ]:
# ==================  CHOOSE THE ARM  ==================
# "lora_fp16" -> fp16 base, LoRA
# "qlora_nf4" -> 4-bit NF4 base, QLoRA
ARM = "lora_fp16"
# =====================================================
assert ARM in ("lora_fp16", "qlora_nf4")
QUANT = "nf4" if ARM == "qlora_nf4" else "none"
print("arm:", ARM, "| quantization:", QUANT)


In [ ]:
# Install the project (git clone; token-authenticated if the repo is private).
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/text2sql-lora-sft")


def authenticated(url: str) -> str:
    token = globals().get("GITHUB_TOKEN")
    if not token or not url.startswith("https://github.com/"):
        return url
    return url.replace("https://", f"https://x-access-token:{token}@", 1)


def find_attached_repo():
    for candidate in Path("/kaggle/input").rglob("pyproject.toml"):
        try:
            if "text2sql-lora" in candidate.read_text(encoding="utf-8"):
                return candidate.parent
        except Exception:
            continue
    return None


if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

cloned = False
try:
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, authenticated(REPO_URL), str(REPO_DIR)],
        capture_output=True, timeout=180, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.replace(str(globals().get("GITHUB_TOKEN", "\0")), "***")[:400])
    cloned = True
    subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], cwd=str(REPO_DIR), capture_output=True)
    print("cloned", REPO_URL)
except Exception as exc:
    print(f"clone failed ({exc}); trying attached dataset")

if not cloned:
    attached = find_attached_repo()
    if attached is None:
        raise SystemExit("Repo not found. Make REPO_URL reachable or attach it as a Dataset.")
    shutil.copytree(attached, REPO_DIR)

sys.path.insert(0, str(REPO_DIR / "src"))
!pip install -q peft bitsandbytes accelerate

# Kaggle preinstalls torchao 0.10.0. PEFT's LoRA dispatcher calls is_torchao_available(),
# which RAISES on a version below its minimum rather than skipping -- which aborts adapter
# injection (ImportError deep inside get_peft_model). We never use torchao (it is an
# optional quantization backend, unrelated to our NF4/bitsandbytes path), so remove it and
# let the check return False cleanly.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)
import importlib

importlib.invalidate_caches()
for _mod in list(sys.modules):
    if _mod == "torchao" or _mod.startswith("torchao."):
        del sys.modules[_mod]
try:
    import peft.import_utils as _iu

    if hasattr(_iu.is_torchao_available, "cache_clear"):
        _iu.is_torchao_available.cache_clear()
    print("torchao available to peft:", _iu.is_torchao_available())  # expect False
except Exception as _e:
    print("torchao check note:", _e)

print("repo ready at", REPO_DIR)

In [ ]:
from pathlib import Path
import importlib
import finqa_lora.run as finqa_run
importlib.reload(finqa_run)

OUT = Path(f"/kaggle/working/artifacts/finqa/{ARM}")
DATA = Path("/kaggle/working/data")

# SMOKE: 5 steps, 8 test questions. Surfaces OOM/dtype errors in minutes. Numbers are meaningless.
finqa_run.run(out_dir=Path("/kaggle/working/smoke"), data_dir=DATA, quantization=QUANT,
              test_limit=8, max_steps=5, generation_batch_size=4)
print("smoke OK -- ignore the scores")


In [ ]:
import gc
gc.collect(); torch.cuda.empty_cache()

# THE REAL ARM. Everything except base precision is identical between arms.
payload = finqa_run.run(out_dir=OUT, data_dir=DATA, quantization=QUANT)


In [ ]:
print("ARM", ARM)
print("  zero-shot ROUGE-1:", round(payload["base_zero_shot"]["rouge1"], 4))
print("  tuned     ROUGE-1:", round(payload["tuned"]["rouge1"], 4))
print("  gain 95% CI      :", payload["rouge1_gain_bootstrap"])
print("  train time (min) :", round(payload["train_seconds"] / 60, 1))
print("  peak VRAM (GB)   :", round(payload["manifest"].get("peak_vram_gb", float("nan")), 2))


In [ ]:
import shutil
shutil.rmtree(OUT / "training", ignore_errors=True)
archive = shutil.make_archive(f"/kaggle/working/{ARM}", "zip", OUT)
print(archive, f"{Path(archive).stat().st_size / 1e6:.1f} MB")
# Unzip locally into artifacts/finqa/<ARM>/ for both arms, then:
#   python scripts/compare_arms.py artifacts/finqa/lora_fp16 artifacts/finqa/qlora_nf4
